# Tau2-Bench: Online Env Splits & Trace Dataset Comparison

**Goal**: Document the relationship between the tau2-bench online environment
(`src/act_prm/environments/tau2bench/`) and the reasoning trace dataset
(`mzio/aprm-amityco_apigen_tau_bench_split_turn`).

## Key Finding

**The trace dataset and tau2 online env cannot be aligned at the task level.**
The traces were generated from `amityco/apigen-tau-bench-split-turn` using
different database states than the verified tau2 tasks, so reservation codes,
user details, and other scenario-specific data don't match. There is no
`task_id` field in the traces.

**However**, this is fine for our use case:
- The online env runs *fresh* interactions with a user simulator (new conversations each time)
- The traces are *pre-generated* conversations from a different data source
- No data leakage is possible since the actual conversation content differs

## Changes Made

1. **Fixed online env** (`Tau2BenchEnv._init_data()`): Now uses tau2's canonical
   `train`/`test` splits from `split_tasks.json` instead of arbitrary index-based splitting.
2. **Updated configs**: `airline.yaml` (30 train / 20 test), `retail.yaml` (74 train / 40 test)
3. **Trace configs** (`act_prm/taubench_*`, `act_lm/taubench_*`): Split independently
   via numpy shuffle (act_prm) or `frac_train_tasks` (act_lm), with `data_seed=0`.

In [ ]:
from datasets import load_dataset
from collections import defaultdict

## 1. Tau2 Online Environment Task Splits

The online env loads tasks via `tau2.run.get_tasks()`. With `task_split=None` (default
in configs), it now loads train and test splits separately using tau2's canonical
`split_tasks.json`.

**Requires** `TAU2_DATA_DIR` to be set (e.g., on hazy1).

```
airline split=None: 50 total tasks (IDs 0-49)
airline split=train: 30 tasks (IDs: 0,1,3,4,5,7,9,10,11,12,14,15,17,20,21,23,27,28,33,34,36,38,39,40,41,42,43,46,47,49)
airline split=test:  20 tasks (IDs: 2,6,8,13,16,18,19,22,24,25,26,29,30,31,32,35,37,44,45,48)

retail split=None: 114 total tasks (IDs 0-113)
retail split=train: 74 tasks
retail split=test:  40 tasks
```

Note: Task IDs are **non-contiguous** within splits — this is why the old
index-based splitting was wrong.

## 2. Trace Dataset Structure

In [ ]:
traces = load_dataset('mzio/aprm-amityco_apigen_tau_bench_split_turn')
print('Trace dataset splits:')
for name, ds in traces.items():
    uids = sorted(set(ds['unique_data_sample_id']))
    gen_ids = sorted(set(ds['generation_id']))
    print(f'  {name}: {len(ds)} rows, {len(uids)} UIDs, '
          f'uid range [{min(uids)}, {max(uids)}], gen_ids={gen_ids}')

## 3. Why Traces Can't Be Mapped to Tau2 Task IDs

The trace dataset was generated from `amityco/apigen-tau-bench-split-turn`,
which contains conversations generated by running tau2 tasks with different
models. However:

1. **No `task_id` field**: The source dataset has no metadata linking
   conversations to specific tau2 task IDs.

2. **Different database states**: The tau2 verified dataset uses different
   reservation codes, user details, etc. than the `apigen` source.
   - Tau2 task 0: reservation `EHGLP3`, user `emma_kim_9957`
   - Trace uid 0: reservation `0U4NPP` (different code!)

3. **Unique first messages**: 1,581 unique first user messages for 1,586
   airline UIDs — the user simulator generates different text each run,
   so message-based matching fails.

4. **Shared identifiers**: Tau2 tasks reuse user names/IDs across tasks,
   making identifier-based matching ambiguous.

In [ ]:
# Demonstrate: first user messages are nearly all unique
ds = traces['airline']
uid_to_first_msg = {}
for i in range(len(ds)):
    row = ds[i]
    if row['timestep'] == 0:
        uid = row['unique_data_sample_id']
        user_msgs = [m['content'] for m in row['state'] if m['role'] == 'user']
        if user_msgs and uid not in uid_to_first_msg:
            uid_to_first_msg[uid] = user_msgs[0]

msg_to_uids = defaultdict(list)
for uid, msg in uid_to_first_msg.items():
    msg_to_uids[msg].append(uid)

print(f'Airline: {len(uid_to_first_msg)} UIDs, {len(msg_to_uids)} unique first messages')
print(f'-> Nearly 1:1 — user simulator generates different text each run')
print(f'\nSample first messages:')
for uid in [0, 1, 2]:
    print(f'  uid={uid}: {uid_to_first_msg[uid][:120]}...')

## 4. Config Comparison: Online Env vs Trace Environments

| Aspect | Online Env (`tau2bench/`) | Act-PRM (`act_prm/taubench_*`) | Act-LM (`act_lm/taubench_*`) |
|--------|--------------------------|-------------------------------|------------------------------|
| **Data source** | Live tau2 tasks + user simulator | Pre-generated traces from `amityco/apigen-tau-bench-split-turn` | Same traces |
| **Splitting** | tau2's `split_tasks.json` (canonical) | numpy shuffle by `data_seed` on `unique_data_sample_id` | `frac_train_tasks` with `data_seed` |
| **Airline train/test** | 30 / 20 tau2 tasks | 1576 / 10 trace UIDs | 95% / 5% of 1586 UIDs |
| **Retail train/test** | 74 / 40 tau2 tasks | 3400 / 10 trace UIDs | 80% / 20% of 3410 UIDs |
| **System prompt** | Domain policy from tau2 + respond_user instruction | Thought-inference prompt | Tool-calling prompt |
| **Observations** | Live tool results from tau2 orchestrator | Pre-recorded in traces | Pre-recorded in traces |
| **User messages** | Generated by user simulator LLM | Pre-recorded in traces | Pre-recorded in traces |
| **Task alignment** | N/A | Independent (no task-level mapping) | Independent |

### System Prompt Differences

| Environment | System Prompt |
|-------------|---------------|
| Online (`tau2bench/`) | `"You are a helpful customer service agent."` + `<policy>` block + respond_user instruction |
| Act-PRM traces | `"You are a helpful assistant that infers reasoning thoughts behind your own observed actions."` |
| Act-LM traces | `"You are a helpful tool-calling assistant."` |

The online env's system prompt includes the full domain policy (airline/retail rules),
while the trace configs use generic prompts. The traces themselves contain the full
policy in each row's `system_prompt` column.

In [ ]:
# Show system prompt from a trace row
ds = traces['airline']
row = ds[0]
sys_prompt = row['system_prompt']
print(f'Trace system_prompt (first 500 chars):')
print(sys_prompt[:500])
print(f'...')
print(f'\nTotal length: {len(sys_prompt)} chars')

## 5. Summary

- **Online env splits are now correct**: Uses tau2's canonical `split_tasks.json`
  (airline: 30 train / 20 test, retail: 74 train / 40 test).
- **Trace splits are independent**: No task-level mapping exists between traces
  and tau2 tasks, so trace configs split by `data_seed` independently.
- **No data leakage risk**: Online env generates fresh conversations via user
  simulator; traces use pre-generated conversations from a different data source
  with different database states.
- **System prompts differ**: Online env includes full domain policy; trace configs
  use task-specific prompts (thought inference for act_prm, tool-calling for act_lm).